In [5]:
!pip install datasets scikit-learn -q

In [6]:
from datasets import load_dataset
import torch

ds_test = load_dataset("dinushiTJ/nz_research_commons_inference_results", split="train", verification_mode="no_checks")
display(ds_test)

README.md:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3318 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse', 'FTEmbGemZeroLabel', 'FTEmbGemZeroID', 'FTEmbGemZeroScore', 'FT4BLabel', 'FT4BResponse'],
    num_rows: 3318
})

In [7]:
print(ds_test.column_names)

['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse', 'FTEmbGemZeroLabel', 'FTEmbGemZeroID', 'FTEmbGemZeroScore', 'FT4BLabel', 'FT4BResponse']


In [8]:
print(ds_test.features)

{'title': Value('string'), 'authors': Value('string'), 'subjects': Value('string'), 'abstract': Value('string'), 'text': Value('string'), 'record_id_hash': Value('string'), 'prompt': Value('string'), 'token_count': Value('int64'), 'classification_label': Value('int64'), 'classification_reason': Value('string'), 'year': Value('string'), 'classification_confidence': Value('float64'), 'embedding_text': Value('string'), 'embedding_token_count': Value('int64'), 'BaseEmbGemZeroLabel': Value('string'), 'BaseEmbGemZeroID': Value('int64'), 'BaseEmbGemZeroScore': Value('float64'), 'Base4BResponse': Value('string'), 'FTEmbGemZeroLabel': Value('string'), 'FTEmbGemZeroID': Value('int64'), 'FTEmbGemZeroScore': Value('float64'), 'FT4BLabel': Value('string'), 'FT4BResponse': Value('string')}


In [9]:
display(ds_test[0]['FT4BResponse'])

'"{\\"label\\": \\"non_maori_origin\\", \\"confidence\\": 1.0, \\"reason\\": \\"The research is explicitly set in Papua New Guinea (PNG) and focuses on the experiences of female students within a Pacific Island cultural context. It has no connection to Māori people, culture, language, or New Zealand.\\"}"'

In [10]:
display(ds_test[0]['FT4BLabel'])

'non_maori_origin'

In [11]:
# import json

# def extract_label_from_ft4b_response(response_string):
#     """
#     Extracts the classification label from the FT4BResponse string.
#     Assumes the label might be a JSON field or directly present in the text.
#     """
#     if not isinstance(response_string, str):
#         return None

#     # Strip the markdown code block delimiters if present
#     cleaned_response_string = response_string.strip()
#     if cleaned_response_string.startswith("```json") and cleaned_response_string.endswith("```"):
#         cleaned_response_string = cleaned_response_string[len("```json"):].strip()
#         cleaned_response_string = cleaned_response_string[:-len("```")].strip()

#     # Try to parse as JSON first
#     try:
#         response_json = json.loads(cleaned_response_string) # Use the cleaned string here
#         # Check if the parsed JSON is a dictionary before trying to access keys
#         if isinstance(response_json, dict):
#             if 'label' in response_json:
#                 return response_json['label']
#         # If it's not a dict, but a simple string (e.g., '"maori_origin"'),
#         # we can return it directly if it matches expected labels.
#         elif isinstance(response_json, str):
#             if response_json.lower() in ['maori_origin', 'non_maori_origin']:
#                 return response_json
#     except json.JSONDecodeError:
#         pass # Not a JSON string (even after cleaning), proceed to text search

#     # If not JSON or key not found in JSON (or JSON was a simple string not matching expected labels),
#     # search for known labels in the original text. Prioritize 'non_maori_origin' if both are present.
#     if 'non_maori_origin' in response_string.lower():
#         return 'non_maori_origin'
#     elif 'maori_origin' in response_string.lower():
#         return 'maori_origin'

#     return None # No label found

Now, let's apply this function to the `FT4BResponse` column and see the extracted labels. We'll add a new column `extracted_label` to our dataset for this.

In [12]:
# # Apply the function to the 'FTBResponse' column
# ds_test = ds_test.map(lambda example: {'FT4BLabel': extract_label_from_ft4b_response(example['FT4BResponse'])})

# # Display the original response and the extracted label for the first few examples
# print("Original FT4BResponse and Extracted Label for the first 5 examples:")
# for i in range(5):
#     print(f"Example {i}:")
#     print(f"  FT4BResponse: {ds_test[i]['FT4BResponse']}")
#     print(f"  FT4BLabel: {ds_test[i]['FT4BLabel']}")
#     print("-" * 30)

In [13]:
import pandas as pd

# Convert the relevant parts of the dataset to a pandas DataFrame for easier filtering
df_filtered = pd.DataFrame({
    'FT4BResponse': ds_test['FT4BResponse'],
    'FT4BLabel': ds_test['FT4BLabel']
})

# Filter for rows where FT4BLabel is not 'maori_origin' or 'non_maori_origin' (including None)
non_standard_labels = df_filtered[~df_filtered['FT4BLabel'].isin(['maori_origin', 'non_maori_origin'])]

print(f"Displaying {len(non_standard_labels)} records with non-standard FT4BLabel values:")
display(non_standard_labels)

Displaying 0 records with non-standard FT4BLabel values:


,FT4BResponse,FT4BLabel


In [14]:
import pandas as pd

# Convert the full ds_test dataset to a pandas DataFrame
df_full = ds_test.to_pandas()

# Get the indices of the non_standard_labels DataFrame
# These are the original indices from the ds_test dataset
non_standard_indices = non_standard_labels.index

# Filter the full DataFrame using these indices
df_all_columns_non_standard = df_full.loc[non_standard_indices]

print(f"Displaying all columns for the {len(df_all_columns_non_standard)} records with non-standard FT4BLabel values:")
display(df_all_columns_non_standard)

Displaying all columns for the 0 records with non-standard FT4BLabel values:


,title,authors,subjects,abstract,text,record_id_hash,prompt,token_count,classification_label,classification_reason,...,embedding_token_count,BaseEmbGemZeroLabel,BaseEmbGemZeroID,BaseEmbGemZeroScore,Base4BResponse,FTEmbGemZeroLabel,FTEmbGemZeroID,FTEmbGemZeroScore,FT4BLabel,FT4BResponse


In [15]:
none_count = 0
maori_count = 0
non_maori_count = 0
other_count = 0

for label in ds_test['FT4BLabel']:
    if label is None:
        none_count += 1
    elif label == 'maori_origin':
        maori_count += 1
    elif label == 'non_maori_origin':
        non_maori_count += 1
    else:
        other_count += 1

print(f"Count of 'maori_origin': {maori_count}")
print(f"Count of 'non_maori_origin': {non_maori_count}")
print(f"Count of None values: {none_count}")
print(f"Count of other (unrecognized) labels: {other_count}")

total_count = maori_count + non_maori_count + none_count + other_count
print(f"Total count: {total_count}")

if none_count > 0:
    print("\nThere are None values in the 'FT4BLabel' column, indicating that some labels could not be extracted.")
if other_count > 0:
    print("There are 'other' (unrecognized) labels, which might indicate an issue with label extraction or unexpected label values.")

Count of 'maori_origin': 360
Count of 'non_maori_origin': 2958
Count of None values: 0
Count of other (unrecognized) labels: 0
Total count: 3318


### Dataset Evaluation

Now, let's evaluate the dataset using the `classification_label` as the true labels and `FTEmbGemZeroLabel` as the predicted labels.

In [16]:
import pandas as pd

df_check = pd.DataFrame({
    "classification_label": ds_test["classification_label"],
    "BaseEmbGemZeroLabel": ds_test["BaseEmbGemZeroLabel"],
    "BaseEmbGemZeroID": ds_test["BaseEmbGemZeroID"],
    "FTEmbGemZeroLabel": ds_test["FTEmbGemZeroLabel"],
    "FTEmbGemZeroID": ds_test["FTEmbGemZeroID"],
    "FT4BLabel": ds_test["FT4BLabel"],
})

print("Base embedding label-ID pairs:")
print(
    df_check[["BaseEmbGemZeroLabel", "BaseEmbGemZeroID"]]
    .drop_duplicates()
    .sort_values("BaseEmbGemZeroID")
)

print("\nFine-tuned embedding label-ID pairs:")
print(
    df_check[["FTEmbGemZeroLabel", "FTEmbGemZeroID"]]
    .drop_duplicates()
    .sort_values("FTEmbGemZeroID")
)

Base embedding label-ID pairs:
  BaseEmbGemZeroLabel  BaseEmbGemZeroID
7        maori_origin                 0
0    non_maori_origin                 1

Fine-tuned embedding label-ID pairs:
  FTEmbGemZeroLabel  FTEmbGemZeroID
7      maori_origin               0
0  non_maori_origin               1


In [17]:
print(
    df_check[["FTEmbGemZeroLabel", "FTEmbGemZeroID"]]
    .drop_duplicates()
    .sort_values("FTEmbGemZeroID")
    .to_string(index=False)
)

FTEmbGemZeroLabel  FTEmbGemZeroID
     maori_origin               0
 non_maori_origin               1


In [ ]:
# from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

# # ClassLabel feature
# class_label = ds_test.features["classification_label"]

# # Ground truth IDs
# true_labels_id = ds_test["classification_label"]

# # Ground truth strings
# true_labels = [
#     class_label.int2str(label_id)
#     for label_id in true_labels_id
# ]

# # Predicted strings
# pred_labels = ds_test["FT4BLabel"]

# # Predicted IDs
# predicted_labels_id = [
#     class_label.str2int(label)
#     for label in pred_labels
# ]

# print(f"Unique true labels: {set(true_labels)}")
# print(f"Unique true labels (IDs): {set(true_labels_id)}")

# print(f"Unique predicted labels: {set(pred_labels)}")
# print(f"Unique predicted labels (IDs): {set(predicted_labels_id)}")

In [18]:
import pandas as pd

df = pd.DataFrame(ds_test)

print(df["FT4BLabel"].value_counts(dropna=False))

FT4BLabel
non_maori_origin    2958
maori_origin         360
Name: count, dtype: int64


In [ ]:
# label_names = class_label.names
# label_to_id = {name: i for i, name in enumerate(label_names)}

In [20]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import pandas as pd

df = pd.DataFrame(ds_test)

# Confirmed mapping from FTEmbGemZeroLabel + FTEmbGemZeroID
label_to_id = {
    "maori_origin": 0,
    "non_maori_origin": 1
}

id_to_label = {
    0: "maori_origin",
    1: "non_maori_origin"
}

label_names = ["maori_origin", "non_maori_origin"]
label_ids = [label_to_id[name] for name in label_names]

# Ground truth IDs
true_labels_id = df["classification_label"].astype(int).tolist()

# Raw FT4B predicted labels
pred_labels_raw = (
    df["FT4BLabel"]
    .astype(str)
    .str.strip()
    .tolist()
)

# Convert predictions to IDs
# Invalid outputs are counted as wrong by forcing the opposite true label
predicted_labels_id = []
invalid_rows = []

for idx, (true_id, pred_label) in enumerate(zip(true_labels_id, pred_labels_raw)):
    if pred_label in label_to_id:
        predicted_labels_id.append(label_to_id[pred_label])
    else:
        invalid_rows.append(idx)
        predicted_labels_id.append(1 - true_id)

# Invalid output summary
invalid_count = len(invalid_rows)
total_count = len(true_labels_id)

print("Total records:", total_count)
print("Invalid prediction count:", invalid_count)
print(f"Invalid prediction rate: {invalid_count / total_count * 100:.4f}%")
print(f"Valid prediction coverage: {(total_count - invalid_count) / total_count * 100:.4f}%")

print("\nInvalid label breakdown:")
if invalid_count > 0:
    print(df.loc[invalid_rows, "FT4BLabel"].value_counts(dropna=False))
else:
    print("No invalid labels")

# Calculate Confusion Matrix
conf_matrix = confusion_matrix(
    true_labels_id,
    predicted_labels_id,
    labels=label_ids
)

print("\nConfusion Matrix (task labels only):")
print(conf_matrix)

# Calculate Classification Report
class_report = classification_report(
    true_labels_id,
    predicted_labels_id,
    labels=label_ids,
    target_names=label_names,
    digits=4
)

print("\nClassification Report (task labels only):")
print(class_report)

# Calculate Accuracy
accuracy = accuracy_score(true_labels_id, predicted_labels_id)
print(f"\nAccuracy: {accuracy:.4f}")

# Calculate F1 Score macro and weighted
f1_macro = f1_score(
    true_labels_id,
    predicted_labels_id,
    average="macro"
)

f1_weighted = f1_score(
    true_labels_id,
    predicted_labels_id,
    average="weighted"
)

print(f"F1 Macro: {f1_macro:.4f}")
print(f"F1 Weighted: {f1_weighted:.4f}")

# Combine metrics into dictionary
metrics = {
    "accuracy": accuracy,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted,
    "invalid_prediction_count": invalid_count,
    "invalid_prediction_rate": invalid_count / total_count,
    "valid_prediction_coverage": (total_count - invalid_count) / total_count
}

print(f"\nCombined Metrics: {metrics}")

Total records: 3318
Invalid prediction count: 0
Invalid prediction rate: 0.0000%
Valid prediction coverage: 100.0000%

Invalid label breakdown:
No invalid labels

Confusion Matrix (task labels only):
[[ 321   46]
 [  39 2912]]

Classification Report (task labels only):
                  precision    recall  f1-score   support

    maori_origin     0.8917    0.8747    0.8831       367
non_maori_origin     0.9844    0.9868    0.9856      2951

        accuracy                         0.9744      3318
       macro avg     0.9381    0.9307    0.9343      3318
    weighted avg     0.9742    0.9744    0.9743      3318


Accuracy: 0.9744
F1 Macro: 0.9343
F1 Weighted: 0.9743

Combined Metrics: {'accuracy': 0.9743821579264618, 'f1_macro': 0.9343481593717462, 'f1_weighted': 0.9742739996903046, 'invalid_prediction_count': 0, 'invalid_prediction_rate': 0.0, 'valid_prediction_coverage': 1.0}


### Analyzing Misclassified Examples

Let's examine the rows where the `classification_label` does not match `Base4BLabel` to understand potential areas of misclassification.

In [ ]:
# Find indices where true labels and predicted labels differ
mismatch_indices = [i for i, (true_id, pred_id) in enumerate(zip(true_labels_id, predicted_labels_id)) if true_id != pred_id]

# Get the mismatched examples from the dataset
mismatched_examples = ds_test.select(mismatch_indices)

# Display relevant columns for the first few mismatched examples
print(f"Found {len(mismatch_indices)} misclassified examples. Displaying the first 10:")

# Define the columns to display for better understanding
columns_to_display = ['title', 'abstract', 'classification_label', 'FT4BLabel', 'classification_reason']

# Convert ClassLabel IDs back to names for better readability in the display
mismatched_examples_df = mismatched_examples.to_pandas()
mismatched_examples_df['classification_label_name'] = mismatched_examples_df['classification_label'].map(lambda x: label_names[x])

# Select and display the relevant columns
display(mismatched_examples_df[columns_to_display + ['classification_label_name']].head(10))

Found 85 misclassified examples. Displaying the first 10:


,title,abstract,classification_label,FT4BLabel,classification_reason,classification_label_name
0,Language learning using Pacific picturebooks i...,Māori and Pacific learners in Aotearoa New Zea...,1,maori_origin,The research focuses specifically on the 'Paci...,non_maori_origin
1,How Maui captured the sun: using a MUD for edu...,"MUDs (Multi-User Dungeons) are text-based, mul...",0,non_maori_origin,The research uses a specific Māori oral tradit...,maori_origin
2,“Pushing an irritational button”: Asian psycho...,The monocultural psychology training in Aotear...,1,maori_origin,The primary focus of the research is the exper...,non_maori_origin
3,"Imua: reflections on imua, talanoa-vā and lead...",The appointment of the University of Waikato’s...,1,maori_origin,The research focuses specifically on Pacific l...,non_maori_origin
4,Paremias xeocosmolóxicas de predición do tempo...,This comparative study collects phraseological...,0,non_maori_origin,The research is a comparative study that subst...,maori_origin
5,Physiological effects of aluminium in relation...,"Alum (aluminium sulphate, Al₂(SO₄)₃) dosing is...",1,maori_origin,The research is a physiological and toxicologi...,non_maori_origin
6,Spatial and temporal characteristics of the un...,Marine ecosystems around the world have been i...,0,non_maori_origin,The research is an ecological study of the Mot...,maori_origin
7,Editorial,"The Volume 43(1) editorial of the ""New Zealand...",0,non_maori_origin,The editorial explicitly highlights research w...,maori_origin
8,Poetic lives: Verse biography in Aotearoa New ...,Bringing together a selection of Aotearoa New ...,0,non_maori_origin,The thesis substantively engages with decoloni...,maori_origin
9,Book review: Mates and lovers: a history of ga...,This article reviews the book: “Mates and love...,1,maori_origin,The research is a book review of a general his...,non_maori_origin
